In [6]:

import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
import os

print("\n" + "="*70)
print("DATA MERGING - CLEAN LABELS + ALL FEATURES")
print("="*70)

# ============================================================
# Step 1: 加载clean labels
# ============================================================

print("\nStep 1: Loading clean Y labels...")

try:
    df_labels = pd.read_parquet('gme_final_with_EWS_labels_TIME_LAGGED.parquet')
    df_labels['timestamp'] = pd.to_datetime(df_labels['timestamp'])
    
    # Remove timezone if present
    if df_labels['timestamp'].dt.tz is not None:
        df_labels['timestamp'] = df_labels['timestamp'].dt.tz_localize(None)
    
    print(f"  ✓ Loaded: {df_labels.shape[0]:,} rows × {df_labels.shape[1]} columns")
    print(f"  Date range: {df_labels['timestamp'].min()} to {df_labels['timestamp'].max()}")
    
    # Check EWS labels
    print(f"\n  EWS Labels distribution:")
    for label in ['EWS_5min', 'EWS_15min', 'EWS_30min']:
        if label in df_labels.columns:
            count = df_labels[label].sum()
            pct = count / len(df_labels) * 100
            print(f"    {label}: {int(count):>6,} ({pct:>5.2f}%)")
        else:
            print(f"    {label}: NOT FOUND")
    
except FileNotFoundError:
    print("\n  ✗ ERROR: gme_final_with_EWS_labels_TIME_LAGGED.parquet not found!")
    print("  Please run create_clean_labels_option_B.py first")
    exit(1)

# ============================================================
# Step 2: 加载所有feature files
# ============================================================

print("\n" + "="*70)
print("Step 2: Loading all feature files...")
print("="*70)

feature_files = {
    'cascade': 'cascade_features_5min.parquet',
    'network': 'network_features_5min.parquet',
    'temporal': 'temporal_features_5min.parquet',
    'text': 'text_features_5min.parquet',
    'burstiness': 'burstiness_features_5min.parquet',
    'user_overlap': 'user_overlap_features_5min.parquet',
    'text_duplication': 'text_duplication_features_5min.parquet'
}

feature_data = {}
missing_files = []

for name, filepath in feature_files.items():
    if os.path.exists(filepath):
        df = pd.read_parquet(filepath)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        # Remove timezone
        if df['timestamp'].dt.tz is not None:
            df['timestamp'] = df['timestamp'].dt.tz_localize(None)
        
        feature_data[name] = df
        print(f"  ✓ {name:20} {df.shape[0]:>6,} rows × {df.shape[1]:>3} cols - {filepath}")
    else:
        print(f"  ✗ {name:20} MISSING - {filepath}")
        missing_files.append(filepath)

if missing_files:
    print(f"\n  ⚠️  WARNING: {len(missing_files)} feature files missing!")
    print(f"  The following features will not be included:")
    for f in missing_files:
        print(f"    - {f}")
    print(f"\n  You can still proceed, but some feature groups will be empty.")
    
    response = input("\n  Continue anyway? (y/n): ")
    if response.lower() != 'y':
        print("\n  Aborted. Please ensure all feature files are available.")
        exit(1)

print(f"\n  ✓ Loaded {len(feature_data)} feature files")

# ============================================================
# Step 3: 合并所有features
# ============================================================

print("\n" + "="*70)
print("Step 3: Merging all features with clean labels...")
print("="*70)

# Start with clean labels
merged = df_labels.copy()
print(f"\n  Starting with: {merged.shape[1]} columns (labels + market)")

# Merge each feature set
for name, df_feat in feature_data.items():
    print(f"\n  Merging {name}...")
    
    before_cols = len(merged.columns)
    
    # Only keep features (not timestamp)
    feat_cols = [c for c in df_feat.columns if c != 'timestamp']
    
    # Merge
    merged = merged.merge(
        df_feat[['timestamp'] + feat_cols],
        on='timestamp',
        how='left',
        suffixes=('', f'_{name}')
    )
    
    after_cols = len(merged.columns)
    new_cols = after_cols - before_cols
    
    print(f"    + {new_cols:>2} new features → total: {after_cols}")
    
    # Check merge success
    null_count = merged[feat_cols].isnull().sum().sum()
    if null_count > 0:
        print(f"    ⚠️  {null_count:,} null values after merge")

print(f"\n  Final shape: {merged.shape[0]:,} rows × {merged.shape[1]} columns")

# ============================================================
# Step 4: 识别feature groups
# ============================================================

print("\n" + "="*70)
print("Step 4: Categorizing features into groups...")
print("="*70)

# Exclude columns
exclude_cols = ['timestamp', 'EWS_5min', 'EWS_15min', 'EWS_30min']

# Initialize feature groups
feature_groups = {
    'market': [],
    'cascade': [],
    'network': [],
    'temporal': [],
    'text': [],
    'burstiness': [],
    'user_overlap': [],
    'text_duplication': []
}

# Auto-categorize
for col in merged.columns:
    if col in exclude_cols:
        continue
    
    col_lower = col.lower()
    
    # Coordination features (priority matching)
    if any(x in col_lower for x in ['burstiness', 'sync', 'inter_arrival', 'burst_index']):
        feature_groups['burstiness'].append(col)
    elif any(x in col_lower for x in ['overlap', 'multi_thread', 'coordinated_group', 
                                       'thread_concentration', 'unique_threads', 'cross_thread']):
        feature_groups['user_overlap'].append(col)
    elif any(x in col_lower for x in ['duplicate', 'duplication', 'similarity', 
                                       'unique_text', 'template']):
        feature_groups['text_duplication'].append(col)
    
    # Reddit features
    elif any(x in col_lower for x in ['cascade', 'virality', 'adoption', 'wiener', 'depth', 'breadth']):
        feature_groups['cascade'].append(col)
    elif any(x in col_lower for x in ['degree', 'betweenness', 'pagerank', 'clustering', 
                                       'k_core', 'centrality', 'density', 'transitivity']):
        feature_groups['network'].append(col)
    elif any(x in col_lower for x in ['volume', 'velocity', 'acceleration', 'growth_rate']):
        feature_groups['temporal'].append(col)
    elif any(x in col_lower for x in ['sentiment', 'urgency', 'emoji', 'keyword', 'toxicity']):
        feature_groups['text'].append(col)
    
    # Market features (default)
    else:
        feature_groups['market'].append(col)

print("\nFeature group summary:")
total_features = 0
for group, features in feature_groups.items():
    print(f"  {group:20} {len(features):>3} features")
    total_features += len(features)

print(f"\n  Total features: {total_features}")

# ============================================================
# Step 5: 标准化features (保留Y labels原样)
# ============================================================

print("\n" + "="*70)
print("Step 5: Standardizing features...")
print("="*70)

# Separate labels and features
label_cols = ['timestamp', 'EWS_5min', 'EWS_15min', 'EWS_30min']
feature_cols = [c for c in merged.columns if c not in label_cols]

print(f"\n  Features to standardize: {len(feature_cols)}")

# Fill NaN with 0 (for missing features)
X_features = merged[feature_cols].fillna(0)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# Create standardized DataFrame
merged_scaled = pd.DataFrame(X_scaled, columns=feature_cols, index=merged.index)

# Add back labels
for col in label_cols:
    if col in merged.columns:
        merged_scaled[col] = merged[col].values

print(f"  ✓ Standardization complete")

# ============================================================
# Step 6: 保存结果
# ============================================================

print("\n" + "="*70)
print("Step 6: Saving outputs...")
print("="*70)

os.makedirs('data', exist_ok=True)

# Save raw merged data
print("\n  Saving merged_data_raw.parquet...")
merged.to_parquet('data/merged_data_raw_CLEAN.parquet', index=False)
print(f"    ✓ {merged.shape[0]:,} rows × {merged.shape[1]} columns")

# Save scaled data
print("\n  Saving merged_data_scaled.parquet...")
merged_scaled.to_parquet('data/merged_data_scaled_CLEAN.parquet', index=False)
print(f"    ✓ {merged_scaled.shape[0]:,} rows × {merged_scaled.shape[1]} columns")

# Save feature groups
print("\n  Saving feature_groups.json...")
with open('data/feature_groups_CLEAN.json', 'w', encoding='utf-8') as f:
    json.dump(feature_groups, f, indent=2)
print(f"    ✓ {len(feature_groups)} feature groups")

# Save feature lists
print("\n  Saving individual feature lists...")
for group, features in feature_groups.items():
    if len(features) > 0:
        pd.DataFrame({'feature': features}).to_csv(
            f'data/features_{group}_CLEAN.csv', index=False
        )
        print(f"    ✓ features_{group}_CLEAN.csv ({len(features)} features)")

# ============================================================
# Step 7: 生成诊断报告
# ============================================================

print("\n" + "="*70)
print("Step 7: Generating diagnostic report...")
print("="*70)

report = f"""
DATA MERGING DIAGNOSTIC REPORT (CLEAN VERSION)
Generated: {pd.Timestamp.now()}

{'='*70}
1. INPUT FILES
{'='*70}

Clean Labels:
  File: gme_final_with_EWS_labels_TIME_LAGGED.parquet
  Rows: {len(df_labels):,}
  Date range: {df_labels['timestamp'].min()} to {df_labels['timestamp'].max()}

Feature Files Loaded: {len(feature_data)}
"""

for name, df in feature_data.items():
    report += f"  - {name:20} {df.shape[0]:>6,} rows × {df.shape[1]:>3} cols\n"

report += f"""

{'='*70}
2. MERGED DATASET
{'='*70}

Final shape: {merged.shape[0]:,} rows × {merged.shape[1]} columns

EWS Labels (Clean - No Leakage):
"""

for label in ['EWS_5min', 'EWS_15min', 'EWS_30min']:
    if label in merged.columns:
        count = merged[label].sum()
        pct = count / len(merged) * 100
        report += f"  {label:12} {int(count):>6,} ({pct:>5.2f}%)\n"

report += f"""

{'='*70}
3. FEATURE GROUPS (for Ablation)
{'='*70}

"""

for group, features in feature_groups.items():
    report += f"{group:20} {len(features):>3} features\n"

report += f"\nTotal features: {total_features}\n"

report += f"""

{'='*70}
4. DATA QUALITY
{'='*70}

Missing values: {merged.isnull().sum().sum():,}
Columns with missing values: {(merged.isnull().sum() > 0).sum()}

"""

# Check for high missing columns
high_missing = merged.isnull().sum()[merged.isnull().sum() > 0].sort_values(ascending=False)
if len(high_missing) > 0:
    report += "Top 10 columns with most missing:\n"
    for col, count in high_missing.head(10).items():
        pct = count / len(merged) * 100
        report += f"  {col:40} {int(count):>6,} ({pct:>5.1f}%)\n"

report += f"""

{'='*70}
5. OUTPUT FILES
{'='*70}

day1_output/
├── merged_data_raw_CLEAN.parquet       ← Original merged
├── merged_data_scaled_CLEAN.parquet    ← Standardized (use this!)
├── feature_groups_CLEAN.json           ← Group definitions
└── features_*_CLEAN.csv                ← Feature lists by group

{'='*70}
6. NEXT STEPS
{'='*70}

✓ Data merging complete with CLEAN Y labels
✓ No data leakage (Time-Lagged approach)

Ready for Day 2:
  1. Update Day 2 model scripts to use:
     - DATA_FILE = 'day1_output/merged_data_scaled_CLEAN.parquet'
     - FEATURE_GROUPS_FILE = 'day1_output/feature_groups_CLEAN.json'
  
  2. Run ablation experiments:
     - python day2_model_xgboost.py
     - python day2_model_logistic.py
     - python day2_model_lightgbm.py
  
  3. Expected results:
     - Baseline: 55-65% PR-AUC (not 98%!)
     - Coordination gain: +5-10%

{'='*70}
"""

# Save report
with open('data/merge_diagnostic_CLEAN.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print(report)
print("  ✓ Saved: data/merge_diagnostic_CLEAN.txt")

# ============================================================
# Summary
# ============================================================

print("\n" + "="*70)
print("DATA MERGING COMPLETE!")
print("="*70)

print(f"\nGenerated files:")
print(f"  ✓ merged_data_scaled_CLEAN.parquet  ← Use this for Day 2!")
print(f"  ✓ feature_groups_CLEAN.json")
print(f"  ✓ merge_diagnostic_CLEAN.txt")

print(f"\nDataset summary:")
print(f"  Total rows: {merged.shape[0]:,}")
print(f"  Total features: {total_features}")
print(f"  Clean Y labels: No leakage (Time-Lagged)")

print(f"\n" + "="*70)
print("READY FOR DAY 2 MODELING!")
print("="*70)

print(f"\nNext: Update Day 2 scripts to use:")
print(f"  DATA_FILE = 'day1_output/merged_data_scaled_CLEAN.parquet'")
print(f"  FEATURE_GROUPS_FILE = 'day1_output/feature_groups_CLEAN.json'")


DATA MERGING - CLEAN LABELS + ALL FEATURES

Step 1: Loading clean Y labels...
  ✓ Loaded: 59,146 rows × 33 columns
  Date range: 2019-07-01 13:25:00 to 2021-06-30 23:55:00

  EWS Labels distribution:
    EWS_5min:  1,417 ( 2.40%)
    EWS_15min:  2,565 ( 4.34%)
    EWS_30min:  4,199 ( 7.10%)

Step 2: Loading all feature files...
  ✓ cascade              59,147 rows ×  13 cols - cascade_features_5min.parquet
  ✓ network              77,267 rows ×  14 cols - network_features_5min.parquet
  ✓ temporal             77,267 rows ×  29 cols - temporal_features_5min.parquet
  ✓ text                 77,267 rows ×  11 cols - text_features_5min.parquet
  ✓ burstiness           77,267 rows ×   7 cols - burstiness_features_5min.parquet
  ✓ user_overlap         77,267 rows ×   6 cols - user_overlap_features_5min.parquet
  ✓ text_duplication     77,267 rows ×   7 cols - text_duplication_features_5min.parquet

  ✓ Loaded 7 feature files

Step 3: Merging all features with clean labels...

  Starting wit

OSError: Cannot save file into a non-existent directory: 'day1_output'